# Burn on a remote GPU peer

Run this with the [evcxr](https://github.com/evcxr/evcxr) Rust Jupyter kernel, launched from the
repository root. Start a compute peer first:

```sh
cargo run -p remote-compute-peer -- burn-web              # CPU
cargo run -p remote-compute-peer --features wgpu -- burn-web   # GPU
```

Every tensor operation in the cells below executes on that peer; only printed or displayed values
are read back. No `async` or `#[tokio::main]` is needed because `RemoteNode::bind_blocking` owns its
runtime. Ending a cell in `tensor.show()` renders the matrix as a heatmap table.

In [ ]:
:dep burn = { path = "crates/burn", features = ["extension", "remote-iroh", "flex"] }
:dep remote-notebook = { path = "examples/remote-notebook" }
:dep blake3 = "1"

In [ ]:
use burn::backend::remote::{EndpointAddr, RemoteNode, SecretKey};
use burn::tensor::{Device, Distribution, Tensor};
use remote_notebook::ShowExt;

fn server_endpoint(topic: &str) -> EndpointAddr {
    let hash = blake3::hash(format!("burn-p2p:{topic}").as_bytes());
    EndpointAddr::from(SecretKey::from_bytes(hash.as_bytes()).public())
}

// Connect to the peer. `device` is reused by every later cell.
let node = RemoteNode::bind_blocking().expect("bind");
let device = Device::remote_iroh(&node, server_endpoint("burn-web"), 0);
println!("connected");

In [ ]:
// A tensor created on the peer, rendered as a heatmap table.
let a = Tensor::<2>::random([3, 4], Distribution::Default, &device);
a.clone().show()

In [ ]:
// Matrix multiply, executed on the peer.
let b = Tensor::<2>::random([4, 2], Distribution::Default, &device);
let c = a.clone().matmul(b);
c.clone().show()

In [ ]:
// Reductions, still on the peer.
c.mean().reshape([1, 1]).show()